# 08 — Similar Items and Similar Users Modules

This notebook builds the project similarity modules:

1. **Metadata item similarity** for “more like this” even when collaborative signal is weak.
2. **ALS item similarity** using item embeddings.
3. **ALS user-user similarity** using user embeddings.
4. Demo-ready parquet files for item pages and user pages.

It tries to load saved tuned ALS factor arrays first. If they are not available, it can retrain the best tuned ALS model from processed data.

In [1]:
import os
import sys
import json
import gc
import pickle
import platform
import subprocess
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.12.12
Pandas: 2.3.3
NumPy: 2.0.2


## 1. Optional dependency for retraining ALS

In [2]:
# This notebook can usually run from saved factor arrays without implicit.
# If factor arrays are missing and retraining is enabled, implicit will be imported/installed.

def ensure_implicit_available():
    try:
        from implicit.als import AlternatingLeastSquares
        import implicit
        print("implicit:", implicit.__version__)
        return AlternatingLeastSquares
    except Exception:
        print("implicit is not installed. Installing now...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "implicit", "-q"])
        from implicit.als import AlternatingLeastSquares
        import implicit
        print("implicit:", implicit.__version__)
        return AlternatingLeastSquares

## 2. Environment, paths, and configuration

In [3]:
PROJECT_NAME = "hm-recommender"
RANDOM_SEED = 42

# Demo sizes. Increase later if you want more precomputed rows for the web app.
N_DEMO_ITEMS = 250
N_DEMO_USERS = 50
N_SIMILAR_ITEMS = 20
N_SIMILAR_USERS = 20

# Metadata TF-IDF settings.
TFIDF_MAX_FEATURES = 60_000
TFIDF_MIN_DF = 2
TFIDF_NGRAM_RANGE = (1, 2)

# Use saved ALS factors if available. Retrain only if they are missing.
RETRAIN_ALS_IF_FACTORS_MISSING = True
SAVE_FULL_SIMILARITY_ARTIFACTS = True

# Fallback config from the hyperparameter tuning review.
FALLBACK_BEST_ALS_CONFIG = {
    "model_name": "als_f64_reg0.1_it15_alpha20",
    "factors": 64,
    "alpha": 20.0,
    "regularization": 0.1,
    "iterations": 15,
}

np.random.seed(RANDOM_SEED)


def detect_environment() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None or Path("/kaggle/input").exists() or Path("/kaggle/working").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") is not None:
        return "colab"
    return "local"


ENVIRONMENT = detect_environment()
IN_KAGGLE = ENVIRONMENT == "kaggle"
IN_COLAB = ENVIRONMENT == "colab"

if IN_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
elif IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PROJECT_DIR = Path.cwd() / PROJECT_NAME

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
SIMILARITY_ARTIFACTS_DIR = ARTIFACTS_DIR / "similarity_modules"
SIMILARITY_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [
    PROJECT_DIR,
    ARTIFACTS_DIR,
    Path.cwd(),
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/content/drive/MyDrive"),
]

print("Detected environment:", ENVIRONMENT)
print("PROJECT_DIR:", PROJECT_DIR)
print("SIMILARITY_ARTIFACTS_DIR:", SIMILARITY_ARTIFACTS_DIR)

Detected environment: kaggle
PROJECT_DIR: /kaggle/working/hm-recommender
SIMILARITY_ARTIFACTS_DIR: /kaggle/working/hm-recommender/artifacts/similarity_modules


In [4]:
def find_file(filename: str, search_roots: List[Path] = SEARCH_ROOTS) -> Optional[Path]:
    candidate_subdirs = [
        "",
        "hm-recommender",
        "hm-recommender/data/processed",
        "hm-recommender/artifacts",
        "hm-recommender/artifacts/als_recommender",
        "hm-recommender/artifacts/als_hyperparameter_tuning",
        "hm-recommender/artifacts/als_hyperparameter_tuning/best_model",
        "hm-recommender/artifacts/hybrid_als_popularity",
    ]

    for root in search_roots:
        if not root.exists():
            continue
        for subdir in candidate_subdirs:
            candidate = root / subdir / filename
            if candidate.exists():
                return candidate

    for root in search_roots:
        if not root.exists():
            continue
        try:
            matches = sorted(root.rglob(filename))
        except Exception:
            matches = []
        if matches:
            return matches[0]

    return None


def find_processed_data_dir() -> Path:
    expected_file = "train_interactions_aggregated.parquet"
    found = find_file(expected_file)
    if found is None:
        raise FileNotFoundError("Could not find processed data. Attach hm-processed-data or rerun preprocessing.")
    return found.parent


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    print("Saved:", path)


def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, compression="snappy")
    print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.4f} MB")


def save_pickle(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.4f} MB")


def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)


def print_df_info(df: pd.DataFrame, name: str, show_head: bool = True) -> None:
    print("\n" + name)
    print("-" * 100)
    print("Shape:", df.shape)
    print(f"Memory usage: {memory_usage_mb(df):.2f} MB")
    if show_head:
        display(df.head())

## 3. Load processed data

In [5]:
PROCESSED_DATA_DIR = find_processed_data_dir()
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)

required_files = [
    "articles_processed.parquet",
    "article_id_map.parquet",
    "customer_id_map.parquet",
    "train_interactions.parquet",
    "train_interactions_aggregated.parquet",
]

for filename in required_files:
    path = PROCESSED_DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(path)
    print(f"Found {filename}: {path.stat().st_size / (1024 ** 2):.2f} MB")

articles = pd.read_parquet(PROCESSED_DATA_DIR / "articles_processed.parquet")
article_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "article_id_map.parquet")
customer_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "customer_id_map.parquet")

train_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "train_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "weight"],
)

train_interactions_aggregated = pd.read_parquet(
    PROCESSED_DATA_DIR / "train_interactions_aggregated.parquet",
    columns=["customer_idx", "article_idx", "purchase_count", "implicit_weight"],
)

for df in [train_interactions, train_interactions_aggregated]:
    df["customer_idx"] = df["customer_idx"].astype("int32")
    df["article_idx"] = df["article_idx"].astype("int32")

train_interactions["t_dat"] = pd.to_datetime(train_interactions["t_dat"])
train_interactions_aggregated["implicit_weight"] = train_interactions_aggregated["implicit_weight"].astype("float32")

n_users = int(customer_id_map["customer_idx"].max()) + 1
n_items = int(article_id_map["article_idx"].max()) + 1

print_df_info(articles, "Articles")
print_df_info(train_interactions, "Train interactions")
print_df_info(train_interactions_aggregated, "Train aggregated interactions")
print("n_users:", n_users)
print("n_items:", n_items)

PROCESSED_DATA_DIR: /kaggle/input/datasets/albaky7/hm-recommender/hm-recommender/data/processed
Found articles_processed.parquet: 6.66 MB
Found article_id_map.parquet: 1.20 MB
Found customer_id_map.parquet: 84.84 MB
Found train_interactions.parquet: 134.36 MB
Found train_interactions_aggregated.parquet: 129.63 MB

Articles
----------------------------------------------------------------------------------------------------
Shape: (105542, 26)
Memory usage: 111.79 MB


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,article_idx
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,1
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,2
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",3
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",4



Train interactions
----------------------------------------------------------------------------------------------------
Shape: (22235277, 4)
Memory usage: 424.10 MB


,t_dat,customer_idx,article_idx,weight
0,2018-09-20,2,40179,1.0
1,2018-09-20,7,46302,1.0
2,2018-09-20,7,6386,1.0
3,2018-09-20,198,47416,1.0
4,2018-09-20,198,5944,1.0



Train aggregated interactions
----------------------------------------------------------------------------------------------------
Shape: (19106319, 4)
Memory usage: 291.54 MB


,customer_idx,article_idx,purchase_count,implicit_weight
0,0,99,1.0,1.693147
1,0,16003,2.0,2.098612
2,0,23996,1.0,1.693147
3,0,29516,1.0,1.693147
4,0,30327,1.0,1.693147


n_users: 1371980
n_items: 105542


## 4. Choose demo query items and users

In [6]:
popular_demo_items = (
    train_interactions["article_idx"]
    .value_counts()
    .head(N_DEMO_ITEMS)
    .index
    .astype("int32")
    .to_numpy()
)

active_demo_users = (
    train_interactions["customer_idx"]
    .value_counts()
    .head(N_DEMO_USERS)
    .index
    .astype("int32")
    .to_numpy()
)

print("Demo items:", len(popular_demo_items), popular_demo_items[:10])
print("Demo users:", len(active_demo_users), active_demo_users[:10])

save_json(
    {
        "n_demo_items": int(len(popular_demo_items)),
        "n_demo_users": int(len(active_demo_users)),
        "n_similar_items": int(N_SIMILAR_ITEMS),
        "n_similar_users": int(N_SIMILAR_USERS),
    },
    SIMILARITY_ARTIFACTS_DIR / "similarity_demo_config.json",
)

Demo items: 250 [53892 53893  1713  2236 14240    67  3711 24837 42626 16003]
Demo users: 50 [1018839  281710  969180  950285 1035425  394603   63928  711274  760470
  459261]
Saved: /kaggle/working/hm-recommender/artifacts/similarity_modules/similarity_demo_config.json


## 5. Metadata item similarity module

In [7]:
metadata_columns = [
    "prod_name",
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "department_name",
    "index_name",
    "section_name",
    "garment_group_name",
    "detail_desc",
]
metadata_columns = [col for col in metadata_columns if col in articles.columns]

articles_for_similarity = articles[["article_idx", "article_id"] + metadata_columns].copy()

for col in metadata_columns:
    articles_for_similarity[col] = articles_for_similarity[col].fillna("").astype(str)

articles_for_similarity["combined_item_text"] = (
    articles_for_similarity[metadata_columns]
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Metadata columns used:", metadata_columns)
print_df_info(articles_for_similarity[["article_idx", "article_id", "combined_item_text"]], "Articles for metadata similarity")

metadata_vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    min_df=TFIDF_MIN_DF,
    ngram_range=TFIDF_NGRAM_RANGE,
    stop_words="english",
)

started = pd.Timestamp.now()
item_metadata_matrix = metadata_vectorizer.fit_transform(articles_for_similarity["combined_item_text"])
print("Metadata TF-IDF matrix shape:", item_metadata_matrix.shape)
print("TF-IDF fit time:", pd.Timestamp.now() - started)

metadata_nn = NearestNeighbors(
    n_neighbors=N_SIMILAR_ITEMS + 1,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)
metadata_nn.fit(item_metadata_matrix)

article_idx_to_row = dict(zip(articles_for_similarity["article_idx"].astype(int), range(len(articles_for_similarity))))
row_to_article_idx = articles_for_similarity["article_idx"].to_numpy(dtype=np.int32)

if SAVE_FULL_SIMILARITY_ARTIFACTS:
    save_npz(SIMILARITY_ARTIFACTS_DIR / "item_metadata_tfidf_matrix.npz", item_metadata_matrix)
    save_pickle(metadata_vectorizer, SIMILARITY_ARTIFACTS_DIR / "item_metadata_tfidf_vectorizer.pkl")
    save_json(
        {
            "matrix_shape": item_metadata_matrix.shape,
            "max_features": TFIDF_MAX_FEATURES,
            "min_df": TFIDF_MIN_DF,
            "ngram_range": TFIDF_NGRAM_RANGE,
            "metadata_columns": metadata_columns,
        },
        SIMILARITY_ARTIFACTS_DIR / "metadata_similarity_artifact_info.json",
    )

Metadata columns used: ['prod_name', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'department_name', 'index_name', 'section_name', 'garment_group_name', 'detail_desc']

Articles for metadata similarity
----------------------------------------------------------------------------------------------------
Shape: (105542, 3)
Memory usage: 38.53 MB


,article_idx,article_id,combined_item_text
0,0,0108775015,Strap top Vest top Garment Upper body Solid Bl...
1,1,0108775044,Strap top Vest top Garment Upper body Solid Wh...
2,2,0108775051,Strap top (1) Vest top Garment Upper body Stri...
3,3,0110065001,OP T-shirt (Idro) Bra Underwear Solid Black Cl...
4,4,0110065002,OP T-shirt (Idro) Bra Underwear Solid White Cl...


Metadata TF-IDF matrix shape: (105542, 60000)
TF-IDF fit time: 0 days 00:00:03.934004
Saved: /kaggle/working/hm-recommender/artifacts/similarity_modules/item_metadata_tfidf_vectorizer.pkl | 2.3708 MB
Saved: /kaggle/working/hm-recommender/artifacts/similarity_modules/metadata_similarity_artifact_info.json


In [8]:
def build_metadata_similar_items(query_article_indices: Iterable[int], n_neighbors: int = N_SIMILAR_ITEMS) -> pd.DataFrame:
    rows = []
    query_rows = []
    valid_query_article_indices = []

    for article_idx in query_article_indices:
        article_idx = int(article_idx)
        row_id = article_idx_to_row.get(article_idx)
        if row_id is not None:
            query_rows.append(row_id)
            valid_query_article_indices.append(article_idx)

    if not query_rows:
        return pd.DataFrame()

    distances, indices = metadata_nn.kneighbors(item_metadata_matrix[query_rows], n_neighbors=n_neighbors + 1)

    for query_article_idx, neighbor_rows, neighbor_distances in zip(valid_query_article_indices, indices, distances):
        rank = 0
        for neighbor_row, distance in zip(neighbor_rows, neighbor_distances):
            similar_article_idx = int(row_to_article_idx[int(neighbor_row)])
            if similar_article_idx == query_article_idx:
                continue
            rank += 1
            rows.append(
                {
                    "query_article_idx": query_article_idx,
                    "similar_article_idx": similar_article_idx,
                    "rank": rank,
                    "similarity_score": float(1.0 - distance),
                    "method": "metadata_tfidf_cosine",
                }
            )
            if rank >= n_neighbors:
                break

    return pd.DataFrame(rows)


metadata_similar_items_demo = build_metadata_similar_items(popular_demo_items, N_SIMILAR_ITEMS)

metadata_similar_items_demo = metadata_similar_items_demo.merge(
    article_id_map.rename(columns={"article_idx": "query_article_idx", "article_id": "query_article_id"})[
        ["query_article_idx", "query_article_id"]
    ],
    on="query_article_idx",
    how="left",
)
metadata_similar_items_demo = metadata_similar_items_demo.merge(
    article_id_map.rename(columns={"article_idx": "similar_article_idx", "article_id": "similar_article_id"})[
        ["similar_article_idx", "similar_article_id"]
    ],
    on="similar_article_idx",
    how="left",
)

preview_cols = ["article_idx", "article_id", "prod_name", "product_type_name", "product_group_name", "colour_group_name", "section_name", "garment_group_name"]
preview_cols = [col for col in preview_cols if col in articles.columns]

similar_item_metadata = articles[preview_cols].rename(
    columns={
        "article_idx": "similar_article_idx",
        "article_id": "similar_article_id",
        "prod_name": "similar_prod_name",
        "product_type_name": "similar_product_type_name",
        "product_group_name": "similar_product_group_name",
        "colour_group_name": "similar_colour_group_name",
        "section_name": "similar_section_name",
        "garment_group_name": "similar_garment_group_name",
    }
)
metadata_similar_items_demo = metadata_similar_items_demo.merge(
    similar_item_metadata,
    on=["similar_article_idx", "similar_article_id"],
    how="left",
)

print_df_info(metadata_similar_items_demo, "Metadata similar items demo")
save_parquet(metadata_similar_items_demo, SIMILARITY_ARTIFACTS_DIR / "metadata_similar_items_demo.parquet")


Metadata similar items demo
----------------------------------------------------------------------------------------------------
Shape: (5000, 13)
Memory usage: 2.83 MB


,query_article_idx,similar_article_idx,rank,similarity_score,method,query_article_id,similar_article_id,similar_prod_name,similar_product_type_name,similar_product_group_name,similar_colour_group_name,similar_section_name,similar_garment_group_name
0,53892,53912,1,1.000000,metadata_tfidf_cosine,0706016001,0706016035,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Black,Divided Collection,Trousers
1,53892,53918,2,1.000000,metadata_tfidf_cosine,0706016001,0706016062,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Black,Divided Collection,Trousers
2,53892,53904,3,0.960126,metadata_tfidf_cosine,0706016001,0706016019,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Dark Blue,Divided Collection,Trousers
3,53892,53908,4,0.960126,metadata_tfidf_cosine,0706016001,0706016028,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Dark Blue,Divided Collection,Trousers
4,53892,53896,5,0.960126,metadata_tfidf_cosine,0706016001,0706016006,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Dark Blue,Divided Collection,Trousers


Saved: /kaggle/working/hm-recommender/artifacts/similarity_modules/metadata_similar_items_demo.parquet | 0.1150 MB


## 6. Load or create ALS embedding factors **[Need to fix the als_f64_reg0.05_it15_alpha40 cause the loaded one should be als_f64_reg0.1_it15_alpha20 cause it is the best ALS config that we found from hyperparameter tuning.**

In [9]:
def load_json_if_found(filename: str) -> Optional[dict]:
    path = find_file(filename)
    if path is None:
        return None
    print("Loaded JSON:", path)
    with open(path) as f:
        return json.load(f)


def load_factor_arrays() -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[Path], Optional[Path]]:
    user_path = find_file("user_factors.npy")
    item_path = find_file("item_factors.npy")
    if user_path is not None and item_path is not None:
        print("Loaded saved ALS factors:")
        print("user_factors:", user_path)
        print("item_factors:", item_path)
        return np.load(user_path), np.load(item_path), user_path, item_path
    return None, None, user_path, item_path


user_factors, item_factors, user_factors_path, item_factors_path = load_factor_arrays()

best_als_config = load_json_if_found("best_als_hyperparameters.json")
if best_als_config is None:
    best_als_config = load_json_if_found("best_als_tuning_config.json")
if best_als_config is None:
    best_als_config = FALLBACK_BEST_ALS_CONFIG.copy()

# Normalize config names.
best_als_config["factors"] = int(best_als_config.get("factors", FALLBACK_BEST_ALS_CONFIG["factors"]))
best_als_config["alpha"] = float(best_als_config.get("alpha", FALLBACK_BEST_ALS_CONFIG["alpha"]))
best_als_config["regularization"] = float(best_als_config.get("regularization", FALLBACK_BEST_ALS_CONFIG["regularization"]))
best_als_config["iterations"] = int(best_als_config.get("iterations", FALLBACK_BEST_ALS_CONFIG["iterations"]))
best_als_config.setdefault(
    "model_name",
    f"als_f{best_als_config['factors']}_reg{best_als_config['regularization']}_it{best_als_config['iterations']}_alpha{int(best_als_config['alpha'])}",
)

print("Best ALS config for similarity modules:")
print(json.dumps(best_als_config, indent=2))

Loaded saved ALS factors:
user_factors: /kaggle/input/datasets/albaky7/hm-hybrid-als-popularity/hm-recommender/artifacts/hybrid_als_popularity/base_als_model/user_factors.npy
item_factors: /kaggle/input/datasets/albaky7/hm-hybrid-als-popularity/hm-recommender/artifacts/hybrid_als_popularity/base_als_model/item_factors.npy
Best ALS config for similarity modules:
{
  "model_name": "als_f64_reg0.1_it15_alpha20",
  "factors": 64,
  "alpha": 20.0,
  "regularization": 0.1,
  "iterations": 15
}


In [10]:
if user_factors is None or item_factors is None:
    if not RETRAIN_ALS_IF_FACTORS_MISSING:
        raise FileNotFoundError("Could not find ALS factor arrays and RETRAIN_ALS_IF_FACTORS_MISSING=False.")

    print("ALS factor arrays were not found. Retraining best tuned ALS model for similarity modules...")
    AlternatingLeastSquares = ensure_implicit_available()

    row_indices = train_interactions_aggregated["customer_idx"].to_numpy(dtype=np.int32)
    col_indices = train_interactions_aggregated["article_idx"].to_numpy(dtype=np.int32)
    values = train_interactions_aggregated["implicit_weight"].to_numpy(dtype=np.float32)

    user_items = csr_matrix((values, (row_indices, col_indices)), shape=(n_users, n_items), dtype=np.float32)
    user_items.eliminate_zeros()

    model = AlternatingLeastSquares(
        factors=int(best_als_config["factors"]),
        regularization=float(best_als_config["regularization"]),
        iterations=int(best_als_config["iterations"]),
        alpha=float(best_als_config["alpha"]),
        random_state=RANDOM_SEED,
        use_gpu=False,
        calculate_training_loss=False,
    )
    model.fit(user_items, show_progress=True)

    user_factors = model.user_factors.astype("float32")
    item_factors = model.item_factors.astype("float32")

    np.save(SIMILARITY_ARTIFACTS_DIR / "user_factors.npy", user_factors)
    np.save(SIMILARITY_ARTIFACTS_DIR / "item_factors.npy", item_factors)
    save_json(best_als_config, SIMILARITY_ARTIFACTS_DIR / "als_similarity_config.json")

    del model
    gc.collect()

print("user_factors shape:", user_factors.shape)
print("item_factors shape:", item_factors.shape)

user_factors shape: (1371980, 64)
item_factors shape: (105542, 64)


## 7. ALS item-item similarity module

In [11]:
item_factors = item_factors.astype("float32")
item_factor_norms = np.linalg.norm(item_factors, axis=1)
valid_item_mask = item_factor_norms > 0
valid_item_indices = np.where(valid_item_mask)[0].astype(np.int32)

item_factors_valid = item_factors[valid_item_indices]
item_factors_valid_normalized = normalize(item_factors_valid, norm="l2", axis=1, copy=True)

als_item_nn = NearestNeighbors(
    n_neighbors=N_SIMILAR_ITEMS + 1,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)
als_item_nn.fit(item_factors_valid_normalized)

article_idx_to_valid_row = dict(zip(valid_item_indices.astype(int), range(len(valid_item_indices))))

print("Valid ALS item factor rows:", len(valid_item_indices))

if SAVE_FULL_SIMILARITY_ARTIFACTS:
    np.save(SIMILARITY_ARTIFACTS_DIR / "item_factors_valid_indices.npy", valid_item_indices)
    np.save(SIMILARITY_ARTIFACTS_DIR / "item_factors_valid_normalized.npy", item_factors_valid_normalized.astype("float32"))

Valid ALS item factor rows: 83275


In [12]:
def build_als_similar_items(query_article_indices: Iterable[int], n_neighbors: int = N_SIMILAR_ITEMS) -> pd.DataFrame:
    rows = []
    query_rows = []
    valid_queries = []

    for article_idx in query_article_indices:
        article_idx = int(article_idx)
        row_id = article_idx_to_valid_row.get(article_idx)
        if row_id is not None:
            query_rows.append(row_id)
            valid_queries.append(article_idx)

    if not query_rows:
        return pd.DataFrame()

    distances, indices = als_item_nn.kneighbors(item_factors_valid_normalized[query_rows], n_neighbors=n_neighbors + 1)

    for query_article_idx, neighbor_rows, neighbor_distances in zip(valid_queries, indices, distances):
        rank = 0
        for neighbor_row, distance in zip(neighbor_rows, neighbor_distances):
            similar_article_idx = int(valid_item_indices[int(neighbor_row)])
            if similar_article_idx == query_article_idx:
                continue
            rank += 1
            rows.append(
                {
                    "query_article_idx": query_article_idx,
                    "similar_article_idx": similar_article_idx,
                    "rank": rank,
                    "similarity_score": float(1.0 - distance),
                    "method": "als_item_embedding_cosine",
                }
            )
            if rank >= n_neighbors:
                break

    return pd.DataFrame(rows)


als_similar_items_demo = build_als_similar_items(popular_demo_items, N_SIMILAR_ITEMS)
als_similar_items_demo = als_similar_items_demo.merge(
    article_id_map.rename(columns={"article_idx": "query_article_idx", "article_id": "query_article_id"})[
        ["query_article_idx", "query_article_id"]
    ],
    on="query_article_idx",
    how="left",
)
als_similar_items_demo = als_similar_items_demo.merge(
    article_id_map.rename(columns={"article_idx": "similar_article_idx", "article_id": "similar_article_id"})[
        ["similar_article_idx", "similar_article_id"]
    ],
    on="similar_article_idx",
    how="left",
)
als_similar_items_demo = als_similar_items_demo.merge(
    similar_item_metadata,
    on=["similar_article_idx", "similar_article_id"],
    how="left",
)

print_df_info(als_similar_items_demo, "ALS similar items demo")
save_parquet(als_similar_items_demo, SIMILARITY_ARTIFACTS_DIR / "als_similar_items_demo.parquet")


ALS similar items demo
----------------------------------------------------------------------------------------------------
Shape: (5000, 13)
Memory usage: 2.85 MB


,query_article_idx,similar_article_idx,rank,similarity_score,method,query_article_id,similar_article_id,similar_prod_name,similar_product_type_name,similar_product_group_name,similar_colour_group_name,similar_section_name,similar_garment_group_name
0,53892,53893,1,0.987385,als_item_embedding_cosine,0706016001,0706016002,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Light Blue,Divided Collection,Trousers
1,53892,53894,2,0.976143,als_item_embedding_cosine,0706016001,0706016003,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Dark Blue,Divided Collection,Trousers
2,53892,53902,3,0.970049,als_item_embedding_cosine,0706016001,0706016015,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Dark Grey,Divided Collection,Trousers
3,53892,53896,4,0.966508,als_item_embedding_cosine,0706016001,0706016006,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Dark Blue,Divided Collection,Trousers
4,53892,53904,5,0.915334,als_item_embedding_cosine,0706016001,0706016019,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Dark Blue,Divided Collection,Trousers


Saved: /kaggle/working/hm-recommender/artifacts/similarity_modules/als_similar_items_demo.parquet | 0.1171 MB


## 8. User-user similarity using ALS user embeddings

In [13]:
user_factors = user_factors.astype("float32")
user_factor_norms = np.linalg.norm(user_factors, axis=1)
valid_user_mask = user_factor_norms > 0
valid_user_indices = np.where(valid_user_mask)[0].astype(np.int32)

user_factors_valid = user_factors[valid_user_indices]
user_factors_valid_normalized = normalize(user_factors_valid, norm="l2", axis=1, copy=True)

als_user_nn = NearestNeighbors(
    n_neighbors=N_SIMILAR_USERS + 1,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)
als_user_nn.fit(user_factors_valid_normalized)

customer_idx_to_valid_row = dict(zip(valid_user_indices.astype(int), range(len(valid_user_indices))))

print("Valid ALS user factor rows:", len(valid_user_indices))

if SAVE_FULL_SIMILARITY_ARTIFACTS:
    np.save(SIMILARITY_ARTIFACTS_DIR / "user_factors_valid_indices.npy", valid_user_indices)
    # This file can be large; save only if configured.
    np.save(SIMILARITY_ARTIFACTS_DIR / "user_factors_valid_normalized.npy", user_factors_valid_normalized.astype("float32"))

Valid ALS user factor rows: 1150557


In [14]:
def build_als_similar_users(query_customer_indices: Iterable[int], n_neighbors: int = N_SIMILAR_USERS) -> pd.DataFrame:
    rows = []
    query_rows = []
    valid_queries = []

    for customer_idx in query_customer_indices:
        customer_idx = int(customer_idx)
        row_id = customer_idx_to_valid_row.get(customer_idx)
        if row_id is not None:
            query_rows.append(row_id)
            valid_queries.append(customer_idx)

    if not query_rows:
        return pd.DataFrame()

    distances, indices = als_user_nn.kneighbors(user_factors_valid_normalized[query_rows], n_neighbors=n_neighbors + 1)

    for query_customer_idx, neighbor_rows, neighbor_distances in zip(valid_queries, indices, distances):
        rank = 0
        for neighbor_row, distance in zip(neighbor_rows, neighbor_distances):
            similar_customer_idx = int(valid_user_indices[int(neighbor_row)])
            if similar_customer_idx == query_customer_idx:
                continue
            rank += 1
            rows.append(
                {
                    "query_customer_idx": query_customer_idx,
                    "similar_customer_idx": similar_customer_idx,
                    "rank": rank,
                    "similarity_score": float(1.0 - distance),
                    "method": "als_user_embedding_cosine",
                }
            )
            if rank >= n_neighbors:
                break

    return pd.DataFrame(rows)


als_similar_users_demo = build_als_similar_users(active_demo_users, N_SIMILAR_USERS)
print_df_info(als_similar_users_demo, "ALS similar users demo")
save_parquet(als_similar_users_demo, SIMILARITY_ARTIFACTS_DIR / "als_similar_users_demo.parquet")


ALS similar users demo
----------------------------------------------------------------------------------------------------
Shape: (1000, 5)
Memory usage: 0.10 MB


,query_customer_idx,similar_customer_idx,rank,similarity_score,method
0,1018839,977462,1,0.617626,als_user_embedding_cosine
1,1018839,527242,2,0.602845,als_user_embedding_cosine
2,1018839,865723,3,0.587437,als_user_embedding_cosine
3,1018839,15068,4,0.572817,als_user_embedding_cosine
4,1018839,593933,5,0.568770,als_user_embedding_cosine


Saved: /kaggle/working/hm-recommender/artifacts/similarity_modules/als_similar_users_demo.parquet | 0.0162 MB


## 9. Similar-user taste summaries

In [15]:
taste_columns = ["article_idx"]
for col in ["product_group_name", "garment_group_name", "section_name", "colour_group_name"]:
    if col in articles.columns:
        taste_columns.append(col)

article_taste = articles[taste_columns].copy()
train_with_taste = train_interactions[["customer_idx", "article_idx"]].merge(
    article_taste,
    on="article_idx",
    how="left",
    validate="many_to_one",
)


def top_values_for_user(user_id: int, col: str, n: int = 5) -> List[str]:
    if col not in train_with_taste.columns:
        return []
    values = (
        train_with_taste.loc[train_with_taste["customer_idx"] == int(user_id), col]
        .dropna()
        .astype(str)
        .value_counts()
        .head(n)
        .index
        .tolist()
    )
    return values


def common_values(list_a: List[str], list_b: List[str]) -> List[str]:
    set_b = set(list_b)
    return [value for value in list_a if value in set_b]


summary_rows = []
for row in als_similar_users_demo.head(N_DEMO_USERS * min(N_SIMILAR_USERS, 10)).itertuples(index=False):
    query_user = int(row.query_customer_idx)
    similar_user = int(row.similar_customer_idx)

    query_groups = top_values_for_user(query_user, "garment_group_name", 5)
    similar_groups = top_values_for_user(similar_user, "garment_group_name", 5)
    query_sections = top_values_for_user(query_user, "section_name", 5)
    similar_sections = top_values_for_user(similar_user, "section_name", 5)

    summary_rows.append(
        {
            "query_customer_idx": query_user,
            "similar_customer_idx": similar_user,
            "rank": int(row.rank),
            "similarity_score": float(row.similarity_score),
            "query_top_garment_groups": " | ".join(query_groups),
            "similar_top_garment_groups": " | ".join(similar_groups),
            "common_garment_groups": " | ".join(common_values(query_groups, similar_groups)),
            "query_top_sections": " | ".join(query_sections),
            "similar_top_sections": " | ".join(similar_sections),
            "common_sections": " | ".join(common_values(query_sections, similar_sections)),
        }
    )

similar_user_taste_summary = pd.DataFrame(summary_rows)
print_df_info(similar_user_taste_summary, "Similar user taste summary")
save_parquet(similar_user_taste_summary, SIMILARITY_ARTIFACTS_DIR / "similar_user_taste_summary.parquet")


Similar user taste summary
----------------------------------------------------------------------------------------------------
Shape: (500, 10)
Memory usage: 0.32 MB


,query_customer_idx,similar_customer_idx,rank,similarity_score,query_top_garment_groups,similar_top_garment_groups,common_garment_groups,query_top_sections,similar_top_sections,common_sections
0,1018839,977462,1,0.617626,"Jersey Fancy | Dresses Ladies | Under-, Nightw...",Trousers | Jersey Fancy | Knitwear | Accessori...,"Jersey Fancy | Under-, Nightwear | Knitwear | ...",Womens Everyday Collection | Womens Lingerie |...,Womens Everyday Collection | H&M+ | Womens Lin...,Womens Everyday Collection | Womens Lingerie |...
1,1018839,527242,2,0.602845,"Jersey Fancy | Dresses Ladies | Under-, Nightw...",Jersey Fancy | Dresses Ladies | Knitwear | Blo...,Jersey Fancy | Dresses Ladies | Knitwear,Womens Everyday Collection | Womens Lingerie |...,Womens Everyday Collection | H&M+ | Divided Co...,Womens Everyday Collection | Divided Collectio...
2,1018839,865723,3,0.587437,"Jersey Fancy | Dresses Ladies | Under-, Nightw...",Dresses Ladies | Jersey Fancy | Knitwear | Jer...,Jersey Fancy | Dresses Ladies | Knitwear,Womens Everyday Collection | Womens Lingerie |...,Womens Everyday Collection | Womens Everyday B...,Womens Everyday Collection | Womens Casual
3,1018839,15068,4,0.572817,"Jersey Fancy | Dresses Ladies | Under-, Nightw...",Jersey Fancy | Dresses Ladies | Trousers | Kni...,Jersey Fancy | Dresses Ladies | Knitwear | Tro...,Womens Everyday Collection | Womens Lingerie |...,Womens Everyday Collection | Divided Collectio...,Womens Everyday Collection | Divided Collection
4,1018839,593933,5,0.568770,"Jersey Fancy | Dresses Ladies | Under-, Nightw...",Jersey Fancy | Accessories | Dresses Ladies | ...,Jersey Fancy | Dresses Ladies | Knitwear,Womens Everyday Collection | Womens Lingerie |...,Womens Everyday Collection | Womens Casual | W...,Womens Everyday Collection | Womens Casual


Saved: /kaggle/working/hm-recommender/artifacts/similarity_modules/similar_user_taste_summary.parquet | 0.0307 MB


## 10. Combined similar-items output for item pages

In [16]:
combined_similar_items = pd.concat(
    [metadata_similar_items_demo, als_similar_items_demo],
    ignore_index=True,
)

# Keep both methods because metadata and ALS answer slightly different questions.
combined_similar_items = combined_similar_items.sort_values(
    ["query_article_idx", "method", "rank"],
    ascending=[True, True, True],
).reset_index(drop=True)

print_df_info(combined_similar_items, "Combined similar items demo")
save_parquet(combined_similar_items, SIMILARITY_ARTIFACTS_DIR / "combined_similar_items_demo.parquet")


Combined similar items demo
----------------------------------------------------------------------------------------------------
Shape: (10000, 13)
Memory usage: 5.67 MB


,query_article_idx,similar_article_idx,rank,similarity_score,method,query_article_id,similar_article_id,similar_prod_name,similar_product_type_name,similar_product_group_name,similar_colour_group_name,similar_section_name,similar_garment_group_name
0,0,1,1,0.967917,als_item_embedding_cosine,0108775015,0108775044,Strap top,Vest top,Garment Upper body,White,Womens Everyday Basics,Jersey Basic
1,0,10109,2,0.910952,als_item_embedding_cosine,0108775015,0538699001,V-neck strap top,Vest top,Garment Upper body,Black,Womens Everyday Basics,Jersey Basic
2,0,10111,3,0.877929,als_item_embedding_cosine,0108775015,0538699007,V-neck strap top,Vest top,Garment Upper body,White,Womens Everyday Basics,Jersey Basic
3,0,1512,4,0.864134,als_item_embedding_cosine,0108775015,0355072001,Anita Tank (1),Vest top,Garment Upper body,Black,Divided Basics,Jersey Basic
4,0,1513,5,0.843357,als_item_embedding_cosine,0108775015,0355072002,Anita Tank (1),Vest top,Garment Upper body,White,Divided Basics,Jersey Basic


Saved: /kaggle/working/hm-recommender/artifacts/similarity_modules/combined_similar_items_demo.parquet | 0.2053 MB


## 11. Final summary

In [17]:
similarity_summary = {
    "notebook": "08-similarity-modules.ipynb",
    "purpose": "Build metadata item similarity, ALS item similarity, and ALS user-user similarity modules.",
    "artifact_dir": str(SIMILARITY_ARTIFACTS_DIR),
    "metadata_similar_items_rows": int(len(metadata_similar_items_demo)),
    "als_similar_items_rows": int(len(als_similar_items_demo)),
    "als_similar_users_rows": int(len(als_similar_users_demo)),
    "similar_user_taste_summary_rows": int(len(similar_user_taste_summary)),
    "combined_similar_items_rows": int(len(combined_similar_items)),
    "best_als_config_used": best_als_config,
    "saved_files": sorted([p.name for p in SIMILARITY_ARTIFACTS_DIR.glob("*")]),
}

print(json.dumps(similarity_summary, indent=2, default=str))
save_json(similarity_summary, SIMILARITY_ARTIFACTS_DIR / "similarity_modules_summary.json")

print("Similarity modules completed successfully.")

{
  "notebook": "08-similarity-modules.ipynb",
  "purpose": "Build metadata item similarity, ALS item similarity, and ALS user-user similarity modules.",
  "artifact_dir": "/kaggle/working/hm-recommender/artifacts/similarity_modules",
  "metadata_similar_items_rows": 5000,
  "als_similar_items_rows": 5000,
  "als_similar_users_rows": 1000,
  "similar_user_taste_summary_rows": 500,
  "combined_similar_items_rows": 10000,
  "best_als_config_used": {
    "model_name": "als_f64_reg0.1_it15_alpha20",
    "factors": 64,
    "alpha": 20.0,
    "regularization": 0.1,
    "iterations": 15
  },
  "saved_files": [
    "als_similar_items_demo.parquet",
    "als_similar_users_demo.parquet",
    "combined_similar_items_demo.parquet",
    "item_factors_valid_indices.npy",
    "item_factors_valid_normalized.npy",
    "item_metadata_tfidf_matrix.npz",
    "item_metadata_tfidf_vectorizer.pkl",
    "metadata_similar_items_demo.parquet",
    "metadata_similarity_artifact_info.json",
    "similar_user_tast